In [1]:
%load_ext autoreload
%autoreload 2

# NF binned background template: review of `src/nf_binned_template.py`

Component 01 (cosmic photons), DC4 511 keV imaging binning.
Checks that integrating the NF density over each CDS bin gives a correct binned template,
first on a few pixels, then against the binned NF samples and the simulation from notebook 08.

## 1. Setup

### 1.1 Imports

All imports of the notebook. `set_global_style()` and `%matplotlib inline` go after the cosipy import,
because cosipy installs its own matplotlib style and backend.

In [2]:
import sys
import time
from pathlib import Path

import numpy as np
import torch                                   # thread count for the speed test (step 4)
import healpy as hp                            # sky maps
import astropy.units as u
from astropy.time import Time
from histpy import Histogram                   # notebook 08 reference histograms
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))     # project root, so `src` is importable
from src.nf_binned_template import NFBinnedTemplate, galactic_conversion_matrix   # imports cosipy
from cosipy.spacecraftfile import SpacecraftHistory   # live time / attitudes from the .ori file

from src.plot_style import COLORS, set_global_style
set_global_style()                             # after cosipy

%matplotlib inline




WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



### 1.2 Configuration

Model, orientation file, DC4 511 keV binning (as in `dataio_511.yaml`) and time window.
The reference histograms are the outputs of notebook 08.

In [3]:
# NF model (component 01) and orientation file
MODEL_PATH = Path("../model/nfbackground_part-01_v3-00.fits")
ORI_PATH = Path("../data/DC4_orientation.ori")

# DC4 511 keV imaging binning (dataio_511.yaml)
EM_EDGES_KEV = [509.0, 513.0]          # 1 energy bin
PHI_EDGES_DEG = np.arange(0, 181, 3)   # 60 bins of 3 deg
NSIDE, SCHEME = 16, "ring"             # PsiChi: 3072 pixels

# Time window of the analysis [unix s]
TMIN, TMAX = 1835487300.036206, 1843467254.887687

# Reference histograms from notebook 08 (Em, Phi, PsiChi)
REF_PATHS = {
    "sim_local":    Path("../data/background_components/CosmicPhotons_511keV_DC4binning_local.hdf5"),
    "sim_galactic": Path("../data/background_components/CosmicPhotons_511keV_DC4binning_galactic.hdf5"),
    "nf_local":     Path("../data/NF_background/part-01_full_caffeinated_511keV_DC4binning_local.hdf5"),
    "nf_galactic":  Path("../data/NF_background/part-01_full_caffeinated_511keV_DC4binning_galactic.hdf5"),
}

## 2. Model normalization

Loads the NF model and reads its `counts` and `livetime`, which fix the rate R = counts / livetime.
No integration yet, so the quadrature settings are the cheapest possible.
Result: `model_livetime` equals the live time in [TMIN, TMAX] (6,579,555 s), so N_i = 97,554,902 · P_i.

In [4]:
# [script] Create the template object from src/nf_binned_template.py.
# This loads the NF model once and builds the integration points for every Em and Phi bin.
# Cheapest settings: 2 Em points on each side of 511 keV, 1 Phi point per bin, 1 sky point per pixel (k = 0).
template = NFBinnedTemplate(MODEL_PATH, EM_EDGES_KEV, PHI_EDGES_DEG, NSIDE, SCHEME,
                            n_em_per_bin=2, n_phi_per_bin=1, subpixel_level=0,
                            em_breaks_kev=[511.0])   # split the Em bin at the 511 keV peak

# [script] Numbers the template read from the model file: they set the rate R = counts / livetime
print(f"model version : {template.model_version}")
print(f"model_counts  : {template.model_counts:,.0f}")
print(f"model_livetime: {template.model_livetime:,.1f} s")


model version : 3.00
model_counts  : 97,554,902
model_livetime: 6,579,555.0 s


## 3. Binning vs the notebook 08 histograms

The template must use the same Em, Phi and PsiChi bins as the reference histograms it will be compared with.
Result: all axes match; totals as in notebook 08.

In [5]:
# Open the four notebook 08 histograms (simulation and NF samples, local and Galactic)
refs = {name: Histogram.open(path) for name, path in REF_PATHS.items()}

# For each one: are its Em and Phi edges the template's edges, and is PsiChi the same HEALPix grid?
for name, h in refs.items():
    ax = h.axes["PsiChi"]
    print(f"{name:13s} Em {np.allclose(h.axes['Em'].edges.to_value(u.keV), template.em_edges_kev)} | "
          f"Phi {np.allclose(h.axes['Phi'].edges.to_value(u.deg), template.phi_edges_deg)} | "
          f"nside {ax.nside}, {ax.scheme} | total {float(h.contents.sum()):,.0f}")


sim_local     Em True | Phi True | nside 16, RING | total 162,652
sim_galactic  Em True | Phi True | nside 16, RING | total 162,652
nf_local      Em True | Phi True | nside 16, RING | total 161,582
nf_galactic   Em True | Phi True | nside 16, RING | total 161,584


## 4. Test pixels

The full sky takes hours, so the integral is first tested on 14 pixels (seconds).
They cover the two poles (did not converge in notebook 07), the pixels with the most NF samples,
the equator and mid-latitudes. With ~50 samples per pixel the Poisson error is ~14% per pixel.

In [6]:
# NF samples per local sky pixel: sum the notebook 08 histogram over Em and Phi -> 3072 numbers
nf_sky = np.asarray(refs["nf_local"].project("PsiChi").to_dense().contents)

# Pick the test pixels (RING numbering, like our PsiChi axis)
polar = np.array([0, 1, 2, 3, 3068, 3069, 3070, 3071])                       # first and last ring: the poles
brightest = np.argsort(nf_sky)[::-1][:3]                                      # 3 pixels with most samples
others = hp.ang2pix(NSIDE, np.deg2rad([90.0, 45.0, 135.0]), np.zeros(3))    # equator, mid-lat N and S
PIXELS = np.unique(np.concatenate([polar, brightest, others]))

# Where each pixel is (colatitude 0 = spacecraft zenith) and how many samples it holds
colat_deg = np.rad2deg(hp.pix2ang(NSIDE, PIXELS)[0])
for p, c in zip(PIXELS, colat_deg):
    print(f"pixel {p:4d}  colatitude {c:6.1f} deg  NF samples {nf_sky[p]:6.0f}")

# [script] How many density evaluations the integral of these pixels needs (cost before running)
print(f"cost at the current settings: {template.n_evaluations(PIXELS):,} evaluations")


pixel    0  colatitude    2.9 deg  NF samples     63
pixel    1  colatitude    2.9 deg  NF samples     64
pixel    2  colatitude    2.9 deg  NF samples     70
pixel    3  colatitude    2.9 deg  NF samples     54
pixel  420  colatitude   45.0 deg  NF samples     56
pixel 1440  colatitude   87.6 deg  NF samples     58
pixel 1469  colatitude   87.6 deg  NF samples     99
pixel 1510  colatitude   90.0 deg  NF samples    101
pixel 1562  colatitude   90.0 deg  NF samples     98
pixel 2592  colatitude  135.0 deg  NF samples     23
pixel 3068  colatitude  177.1 deg  NF samples     44
pixel 3069  colatitude  177.1 deg  NF samples     32
pixel 3070  colatitude  177.1 deg  NF samples     41
pixel 3071  colatitude  177.1 deg  NF samples     32
cost at the current settings: 3,360 evaluations


## 5. First background with the script (k = 0)

`get_binned_background` gives expected counts; `pixels=` restricts it to the test pixels (local frame).
Compared with the NF samples: same model, so they should agree within Poisson.
Result: pixels away from the poles and the χ = 0 seam agree; poles (+30–70%) and pixel 1440 (×8 low) do not.

In [7]:
# [script] Expected background counts, local frame, only for the test pixels.
# Inside, get_binned_background (1) integrates the density over every bin of these pixels (compute, k = 0)
# and (2) multiplies P_i by rate x live time read from the orientation file -> counts N_i.
bkg_local = template.get_binned_background("local", ORI_PATH, TMIN, TMAX, pixels=PIXELS)
print("shape:", bkg_local.contents.shape)          # full sky (1, 60, 3072); pixels not computed are NaN

# Counts per test pixel: add up its 60 Phi bins (and the single Em bin)
N_sub = bkg_local.contents[:, :, PIXELS].sum(axis=(0, 1))
n_obs = nf_sky[PIXELS]                             # NF samples in the same pixels (reference)
pull = (n_obs - N_sub) / np.sqrt(N_sub)            # difference in units of the Poisson error

for p, N, n, z in zip(PIXELS, N_sub, n_obs, pull):
    print(f"pixel {p:4d}  integral {N:6.1f}  NF samples {n:4.0f}  pull {z:+.2f}")
print(f"sum: integral {N_sub.sum():.1f}  NF samples {n_obs.sum():.0f}")


14 pixels, k = 0: 3,360 evaluations in 1 chunks of up to 8333 pixels
  chunk 1/1: 14/14 pixels, 0 s, 6,725 evaluations/s
shape: (1, 60, 14)
pixel    0  integral  107.6  NF samples   63  pull -4.30
pixel    1  integral  110.3  NF samples   64  pull -4.41
pixel    2  integral  110.8  NF samples   70  pull -3.88
pixel    3  integral  104.2  NF samples   54  pull -4.92
pixel  420  integral   45.5  NF samples   56  pull +1.55
pixel 1440  integral    7.2  NF samples   58  pull +18.89
pixel 1469  integral   79.8  NF samples   99  pull +2.15
pixel 1510  integral   84.5  NF samples  101  pull +1.79
pixel 1562  integral   89.3  NF samples   98  pull +0.92
pixel 2592  integral   29.9  NF samples   23  pull -1.27
pixel 3068  integral   52.2  NF samples   44  pull -1.14
pixel 3069  integral   51.5  NF samples   32  pull -2.72
pixel 3070  integral   51.3  NF samples   41  pull -1.44
pixel 3071  integral   54.2  NF samples   32  pull -3.02
sum: integral 978.4  NF samples 835


## 6. Convergence with the number of sky points (k)

If the integral is right, adding sky points per pixel (4**k) must change the result less and less.
Result: normal pixels converge already at k = 1. Pixel 1440 (χ ≈ 0 seam) keeps rising, and the poles
go down then up: the density is not smooth at the edges of the flow's domain (hypothesis, to check).

In [8]:
# Same pixels, more sky points per pixel: 4**k points, k = 0, 1, 2, 3 (Em and Phi points unchanged)
N_k = {}
for k in range(4):
    template.compute(PIXELS, subpixel_level=k, verbose=False)               # [script] the integral, at level k
    N_k[k] = template.expected_counts()[:, :, PIXELS].sum(axis=(0, 1))     # [script] counts N = model_counts * P

# One row per pixel: how the counts change with k, next to the NF samples
print("pixel  " + "  ".join(f"k={k:<5d}" for k in N_k) + "  NF samples")
for i, p in enumerate(PIXELS):
    print(f"{p:5d}  " + "  ".join(f"{N_k[k][i]:7.1f}" for k in N_k) + f"  {n_obs[i]:6.0f}")


pixel  k=0      k=1      k=2      k=3      NF samples
    0    107.6     56.9     49.8     55.8      63
    1    110.3     57.8     50.3     56.9      64
    2    110.8     57.9     49.8     56.3      70
    3    104.2     57.6     49.4     56.0      54
  420     45.5     45.7     45.6     45.9      56
 1440      7.2     26.9     37.5     42.9      58
 1469     79.8     79.8     80.0     80.0      99
 1510     84.5     80.5     79.5     79.4     101
 1562     89.3     85.0     83.7     83.6      98
 2592     29.9     29.7     30.0     30.3      23
 3068     52.2     29.7     27.6     31.8      44
 3069     51.5     29.5     27.4     31.6      32
 3070     51.3     30.3     27.5     31.6      41
 3071     54.2     29.8     28.0     32.4      32
